In [54]:
import pyautogui as pyg

# 輸入最左上角寶石中心點座標，以及相鄰寶石中心點的間距
x = 1219
y = 271
gap = 35.5

In [55]:


# 擷取畫面後，讀取 8×8 個寶石中心點的 RGB 顏色
screen = pyg.screenshot()
for row in range(8):
    for col in range(8):
        center_x = x + col * gap
        center_y = y + row * gap
        color = screen.getpixel((center_x, center_y))
        print(f"({row}, {col})（{center_x}, {center_y}）：{color}")

(0, 0)（1219.0, 271.0）：(14, 123, 243)
(0, 1)（1254.5, 271.0）：(232, 120, 48)
(0, 2)（1290.0, 271.0）：(247, 26, 53)
(0, 3)（1325.5, 271.0）：(232, 222, 222)
(0, 4)（1361.0, 271.0）：(234, 123, 49)
(0, 5)（1396.5, 271.0）：(241, 17, 220)
(0, 6)（1432.0, 271.0）：(233, 211, 211)
(0, 7)（1467.5, 271.0）：(241, 17, 221)
(1, 0)（1219.0, 306.5）：(90, 246, 144)
(1, 1)（1254.5, 306.5）：(251, 251, 251)
(1, 2)（1290.0, 306.5）：(252, 27, 57)
(1, 3)（1325.5, 306.5）：(18, 129, 247)
(1, 4)（1361.0, 306.5）：(82, 246, 133)
(1, 5)（1396.5, 306.5）：(243, 16, 243)
(1, 6)（1432.0, 306.5）：(18, 130, 247)
(1, 7)（1467.5, 306.5）：(253, 251, 37)
(2, 0)（1219.0, 342.0）：(249, 249, 249)
(2, 1)（1254.5, 342.0）：(253, 249, 36)
(2, 2)（1290.0, 342.0）：(242, 11, 242)
(2, 3)（1325.5, 342.0）：(79, 244, 128)
(2, 4)（1361.0, 342.0）：(80, 244, 129)
(2, 5)（1396.5, 342.0）：(251, 236, 111)
(2, 6)（1432.0, 342.0）：(77, 244, 124)
(2, 7)（1467.5, 342.0）：(252, 235, 108)
(3, 0)（1219.0, 377.5）：(253, 253, 37)
(3, 1)（1254.5, 377.5）：(248, 14, 248)
(3, 2)（1290.0, 377.5）：(20, 115, 23

In [56]:
# 使用前一個 cell 輸入的左上角座標作為 basex、basey
basex, basey = x, y

def gem_position(r,c):
    """回傳第 r 列、第 c 欄寶石中心點的 (x, y) 座標；列、欄從 0 開始。"""
    return (basex + c * gap, basey + r * gap)

In [57]:
def get_gem_color(r, c, im):
    """回傳截圖 im 中第 (r, c) 顆寶石中心點的 RGB 顏色。"""
    x, y = gem_position(r, c)
    return im.getpixel((round(x), round(y)))

In [58]:
def _can_move(r, c, dr, dc, im):
    """判斷第 (r, c) 顆寶石往指定方向交換後是否會形成三連線。"""
    nr, nc = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    colors = [[get_gem_color(row, col, im) for col in range(8)] for row in range(8)]
    colors[r][c], colors[nr][nc] = colors[nr][nc], colors[r][c]

    def same_color(color1, color2):
        # 中心像素可能有些微色差，以 RGB 各通道差距不超過 40 判定同色。
        return all(abs(a - b) <= 40 for a, b in zip(color1, color2))

    def has_match(row, col):
        color = colors[row][col]
        for match_dr, match_dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                rr = row + match_dr * direction
                cc = col + match_dc * direction
                while (0 <= rr < 8 and 0 <= cc < 8
                       and same_color(colors[rr][cc], color)):
                    count += 1
                    rr += match_dr * direction
                    cc += match_dc * direction
            if count >= 3:
                return True
        return False

    return has_match(r, c) or has_match(nr, nc)

def can_move_left(r, c, im):
    return _can_move(r, c, 0, -1, im)

def can_move_right(r, c, im):
    return _can_move(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_move(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_move(r, c, 1, 0, im)

In [59]:
im=pyg.screenshot()
print(can_move_left(5,4,im))

False


In [60]:
print(gem_position(2,5))
print(gem_position(1,3))

(1396.5, 342.0)
(1325.5, 306.5)


In [61]:
im=pyg.screenshot()
for i in range(8):
    x,y=gem_position(0,i)
    pyg.click(x,y,duration=1)
    print(get_gem_color(0,i,im))

(14, 123, 243)
(232, 120, 48)
(247, 26, 53)
(230, 230, 230)
(234, 123, 49)
(241, 17, 220)
(233, 211, 211)
(242, 15, 242)


In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    moved = False
    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            if can_move_left(r, c, im):
                destination = (r, c - 1)
            elif can_move_right(r, c, im):
                destination = (r, c + 1)
            elif can_move_up(r, c, im):
                destination = (r - 1, c)
            elif can_move_down(r, c, im):
                destination = (r + 1, c)
            else:
                continue

            start_x, start_y = gem_position(r, c)
            end_x, end_y = gem_position(*destination)
            pyg.moveTo(round(start_x), round(start_y))
            pyg.dragTo(round(end_x), round(end_y), duration=0.2, button='left')
            im = pyg.screenshot()
            moved = True

        if time.time() - start_time > 60:
            break

    if not moved:
        im = pyg.screenshot()
        time.sleep(0.1)